# 🔥 TERRA-FIRE: Predictive Wildfire Intelligence Engine
## **Activity 8. Intelligent Segmentation Lab — Unsupervised K-Means Clustering, Centroid Cartography & Regime Discovery**
**Institution:** Universidad Tecnológica El Retoño (UTR)  
**Academic Program:** Information Technologies and Digital Innovation Engineering (`LITD9A`)  
**Subject:** Data Science (Sep–Dec 2026)  
**Teacher:** Dr. Sergio Valadez Godínez  
**Students:** Angel Emilio Gutierrez Lozano (ID: 4341) & Emiliano Betanzos Valtierra (ID: 4795)  
**Official Repository:** [https://github.com/AngelGTZ28/TERRA-FIRE](https://github.com/AngelGTZ28/TERRA-FIRE)

---
### 🎯 Analytical Transition (`Predictive Modeling` $\to$ `Intelligent Segmentation`)
Building directly upon our validated **Analysis-Ready Dataset** (`Activity 3`), **Statistical Group Contrasts** (`Activity 4`), **Correlation & OLS Regression Feature Hierarchy** (`Activity 5`), **12-to-3 Orthogonal PCA Dimensionality Reduction** (`Activity 6`), and **Supervised Binary Classification Benchmark** (`Activity 7`), this notebook executes **Activity 8 (Intelligent Segmentation Lab)** to answer the core unsupervised research question:
> **What natural groupings or segments exist within the Aguascalientes and Sierra Fría spatiotemporal monitoring dataset when analyzed strictly without predefined target labels?**

*Methodological & Domain Scope Note:* Because `terra_fire_spatiotemporal_matrix_v1.parquet` ($N = 18,275 \times 22$) is a structured spatiotemporal tabular feature store where Sentinel-2 spectral indices (`ndmi`) and Copernicus DEM terrain metrics (`elevation_m`, `slope_deg`) are pre-extracted continuous numerical attributes, conditional sections for raw computer-vision pixel/texture image segmentation do not apply. Furthermore, to uphold a strict **unsupervised learning boundary**, the binary target `fire_ignition_48h` is **completely excluded** from the clustering input matrix and evaluated exclusively *post-hoc* in Phases 3 and 4 to validate operational relevance.


---
## 🧭 PHASE 1: SEGMENTATION PROBLEM FORMULATION
### Step 1 — Segmentation Objective Definition
We formulate the unsupervised segmentation problem across the 25 monitoring nodes (`AGS_VOX_01` to `AGS_VOX_25`) over the 2023–2024 hydrological cycle ($N = 18,275$ daily observations) to discover homogeneous pyro-climatic and territorial regimes (**Deliverable 1 — Segmentation Objective Matrix**).


In [1]:
import os, json, joblib
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.spatial.distance import cdist, pdist, squareform
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

RANDOM_STATE = 42
DATA_PATH = '../data/processed/terra_fire_spatiotemporal_matrix_v1.parquet'
df = pd.read_parquet(DATA_PATH)
print(f'[READ-ONLY LOAD] Immutable Dataset Loaded: {df.shape[0]:,} observations x {df.shape[1]} attributes')
print('[DELIVERABLE 1] Segmentation Objective Matrix:')
df_obj


[READ-ONLY LOAD] Immutable Dataset Loaded: 18,275 observations x 22 attributes
[DELIVERABLE 1] Segmentation Objective Matrix:


,Element,Description
0,Population,"18,275 daily spatiotemporal monitoring observations across 25 regional micro-territorial nodes (AGS_VOX_01 to AGS_VOX_25, 10–20 m resolution) spanning the agricultural valleys and Sierra Fría biological reserve in Aguascalientes, Mexico (2023–2024)."
1,Objective,"Discover natural, unsupervised pyro-climatic and territorial risk regimes without predefined labels (excluding fire_ignition_48h) to identify homogeneous micro-territorial segments that complement supervised 48–72h wildfire hazard predictions."
2,Variables,"Primary Input: 3 Kaiser-significant orthogonal Principal Components ['PC1', 'PC2', 'PC3'] (87.87% cumulative variance retained, VIF = 1.00) derived from the 12 core standardized variables (dist_agri_m, dist_roads_m, temp_2m_max_c, rh_2m_min_pct, soil_moist_mean, vpd_max_kpa, ndmi, slope_deg, elevation_m, vpd_72h_max_kpa, precip_72h_sum_mm, days_since_rain)."
3,Expected Value,"Automated territorial regime stratification for CONAFOR & Civil Protection: Differentiating winter/spring cured-biomass agricultural burn risk (NOM-015) from extreme late-spring thermodynamic heatwave apexes, monsoon moisture-suppressed valleys, and high-altitude temperate forest sanctuaries."


### Step 2 — Dataset Preparation & Input Representation Justification
Because $K$-Means minimizes squared Euclidean distances $\sum_{k=1}^K \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|_2^2$, feeding unscaled or highly collinear features distorts cluster geometry:
1. **Z-Score Standardization (`StandardScaler`):** Each of the 12 core quantitative predictors validated in Activity 5 is standardized to $\mu = 0, \sigma = 1$.
2. **Orthogonal PCA Projection (`PCA(n_components=3, random_state=42)`):** In the 12D standardized space, severe multicollinearity ($\text{Max VIF} = 80.10$, $\text{Mean VIF} = 30.74$ across `dist_agri_m`, `dist_roads_m`, `slope_deg`, and `elevation_m`) causes Euclidean distance to quadruple-weight topographic/anthropogenic proximity. Projecting onto the 3 Kaiser-significant Principal Components (`['PC1', 'PC2', 'PC3']`, $\lambda_1 = 4.8918, \lambda_2 = 4.0088, \lambda_3 = 1.6440$) retains **87.87% of total variance** with **perfect orthogonality ($\text{VIF} = 1.00$)** (**Deliverable 2 & Deliverable 3**).


In [2]:
CORE_FEATURES_12 = [
    'dist_agri_m', 'dist_roads_m', 'temp_2m_max_c', 'rh_2m_min_pct',
    'soil_moist_mean', 'vpd_max_kpa', 'ndmi', 'slope_deg',
    'elevation_m', 'vpd_72h_max_kpa', 'precip_72h_sum_mm', 'days_since_rain'
]
scaler = StandardScaler()
X_scaled_12 = pd.DataFrame(scaler.fit_transform(df[CORE_FEATURES_12].astype(float)), columns=CORE_FEATURES_12, index=df.index)
pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca_3 = pd.DataFrame(pca.fit_transform(X_scaled_12), columns=['PC1', 'PC2', 'PC3'], index=df.index)

print(f'[PCA 3D SUBSPACE] Shape: {X_pca_3.shape} | Explained Variance: {pca.explained_variance_ratio_*100} -> Cumulative: {sum(pca.explained_variance_ratio_)*100:.2f}%')
print('\n[DELIVERABLE 2] Clustering Dataset Report:')
display(df_ds_report)
print('\n[DELIVERABLE 3] Segmentation Input Structure Report:')
df_input_struct


[PCA 3D SUBSPACE] Shape: (18275, 3) | Explained Variance: [40.76297185 33.40509675 13.69890187] -> Cumulative: 87.87%

[DELIVERABLE 2] Clustering Dataset Report:
               Dataset Observations                                                                                                                                           Variables
0  Clustering Dataset       18,275  3 Orthogonal PCA Components ['PC1', 'PC2', 'PC3'] (derived from 12 standardized core physical variables; target fire_ignition_48h strictly excluded)

[DELIVERABLE 3] Segmentation Input Structure Report:


,Input Type,Variables / Components,Justification
0,Original Variables,"12 Standardized Core Variables (dist_agri_m, dist_roads_m, temp_2m_max_c, rh_2m_min_pct, soil_moist_mean, vpd_max_kpa, ndmi, slope_deg, elevation_m, vpd_72h_max_kpa, precip_72h_sum_mm, days_since_rain)","Retains 100% of raw physical variance (TSS = 219,300.0) and direct physical units, but suffers from severe multicollinearity (Max VIF = 80.10, Mean VIF = 30.74) which double-counts correlated topographic/anthropogenic axes and degrades Euclidean K-Means Silhouette to 0.2496 at K=4."
1,PCA Components (Selected Primary Input),"3 Orthogonal Principal Components: PC1 (40.76% Atmospheric-Edaphic Desiccation, lambda_1=4.8918), PC2 (33.41% Orographic Altitude vs. Agri Valley, lambda_2=4.0088), PC3 (13.70% Canopy Phenology NDMI & Warm Monsoon, lambda_3=1.6440)","Compresses dimensionality by 75.0% (12 -> 3) while retaining 87.87% of total variance (TSS = 192,692.27) with exact orthogonality (VIF = 1.00). Eliminates collinear Euclidean metric distortion, boosting K-Means Silhouette Score by +21.7% (from 0.2496 to 0.3039 at K*=4) and enabling clean 2D/3D centroid geometry."


---
## ⚙️ PHASE 2: K-MEANS IMPLEMENTATION AND CLUSTER EXPLORATION
### Step 3 — K-Means Configuration, Elbow Method & Silhouette Analysis
We evaluate candidate cluster counts $K \in \{2, 3, 4, 5, 6, 7, 8\}$ using `KMeans(n_clusters=K, init='k-means++', n_init=20, max_iter=300, random_state=42)` on the 3D orthogonal PCA space `X_pca_3` (and benchmark against the 12D standardized space `X_scaled_12`).
- **Selection of Optimal $K^* = 4$:**
  1. **Geometric Elbow Inflection:** From $K=2 \to 3$, WCSS drops by $-40,084.13$ ($-30.02\%$), and from $K=3 \to 4$, WCSS drops by $-19,787.21$ ($-21.18\%$), explaining **61.78%** of total 3D variance. Beyond $K=4$, marginal WCSS reduction flattens abruptly to $-9,029.69$ ($-12.26\%$ at $K=5$).
  2. **Multi-Index Validation Optimum:** At $K^* = 4$, the 3D PCA segmentation achieves **Mean Silhouette Score $= 0.3039$** ($+21.7\%$ higher than 12D raw K-Means at $0.2496$), alongside the **global minimum Davies-Bouldin Index ($1.0474$)** and **global maximum Calinski-Harabasz Index ($9,844.88$)** across all $K \in \{2..8\}$.
  3. **Physical Resolution of `PC3`:** Whereas $K=3$ leaves all three centroids near zero on `PC3` ($+0.11, -0.02, -0.16$), $K^*=4$ unlocks `PC3` (*Canopy Phenology `ndmi` & Warm Monsoon*), separating **Cluster 0 (Dormant Cured-Biomass Valley, $\text{PC3} = -1.19$)** from **Cluster 2 (Extreme Heatwave Desiccation Apex, $\text{PC1} = +2.66, \text{PC3} = +0.68$)** (**Deliverable 4 & Deliverable 5**).


In [3]:
print('[DELIVERABLE 4] K-Means Configuration Matrix (K = 2..8):')
display(df_kmeans_config)
print('\n[DELIVERABLE 5] Cluster Assignment Report (Optimal K* = 4):')
df_cluster_assign


[DELIVERABLE 4] K-Means Configuration Matrix (K = 2..8):
   K                                         Inertia                Silhouette Score
0  2  133,517.11 (30.71% Var | 12D: 159,889.27)  0.2877 (12D: 0.2533 | +13.6%)
1  3    93,432.98 (51.51% Var | 12D: 119,780.71)  0.3208 (12D: 0.2779 | +15.4%)
2  4     73,645.77 (61.78% Var | 12D: 99,558.24)  0.3039 (12D: 0.2496 | +21.7%)
3  5     64,616.08 (66.47% Var | 12D: 89,485.36)  0.2996 (12D: 0.2366 | +26.6%)
4  6     57,045.69 (70.40% Var | 12D: 81,279.14)  0.2936 (12D: 0.2277 | +28.9%)
5  7     50,976.44 (73.55% Var | 12D: 73,240.80)  0.2868 (12D: 0.2479 | +15.7%)
6  8     46,179.73 (76.03% Var | 12D: 67,164.17)  0.2821 (12D: 0.2492 | +13.2%)

[DELIVERABLE 5] Cluster Assignment Report (Optimal K* = 4):


,Cluster,Number of Observations
0,Cluster 0 (Dormant Cured Valley),"5,815 (31.82%)"
1,Cluster 1 (Sierra Fria Sanctuary),"3,042 (16.65%)"
2,Cluster 2 (Extreme Heatwave Apex),"4,695 (25.69%)"
3,Cluster 3 (Monsoon Hydrated Valley),"4,723 (25.84%)"
4,Total Population (N),"18,275 (100.00%)"


### Step 4 — Cluster Visualization Dashboard (`fig34` – `fig37`)
Below we render the four high-resolution (300 DPI) visual analytics dashboards examining Elbow/Silhouette optimality (`fig34`), 2D/3D PCA cluster geometry and centroids (`fig35`), standardized centroid profiles and pairwise feature projections (`fig36`), and cluster size, ecological fuel composition, and post-hoc wildfire hazard distributions (`fig37`).

![Figure 34: K-Means Elbow and Silhouette Analysis](../reports/figures/fig34_kmeans_elbow_and_silhouette.png)

![Figure 35: 2D and 3D PCA Cluster Geometry and Centroids](../reports/figures/fig35_pca_clusters_2d_and_3d_centroids.png)

![Figure 36: Cluster Centroid Profiles and Pairwise Feature Projections](../reports/figures/fig36_cluster_centroid_profiles_and_pairplot.png)

![Figure 37: Cluster Size Distributions, Fuel Composition and Post-Hoc Hazard](../reports/figures/fig37_cluster_distributions_and_domain_characterization.png)


---
## 🔬 PHASE 3: CLUSTER INTERPRETATION AND VALIDATION
### Step 5 — Cluster Characterization
We characterize each of the $K^* = 4$ clusters across the 3 orthogonal PCA axes (`PC1`, `PC2`, `PC3`), the 12 physical variables (both Z-scores and original measurement units), seasonal prevalence, and post-hoc ecological/wildfire profiles (**Deliverable 6 — Cluster Characterization Matrix**).


In [4]:
print('[DELIVERABLE 6] Cluster Characterization Matrix (K* = 4):')
df_characterization


[DELIVERABLE 6] Cluster Characterization Matrix (K* = 4):


,Cluster,Main Characteristics,Interpretation
0,"Cluster 0 (Dormant Cured-Biomass Agricultural Valley — N = 5,815, 31.82%)","Centroid: PC1 = +0.26, PC2 = -0.60, PC3 = -1.19 (Pyro Index = +0.86). Lowest foliar moisture (ndmi = -0.007, Z = -0.77), cool-dry winter/early-spring weather (temp_2m_max_c = 22.17°C, rh_2m_min_pct = 26.09%, vpd_max_kpa = 1.66 kPa, precip_72h_sum_mm = 0.68 mm, days_since_rain = 15.2 d), low agricultural valley elevation (2,139.2 m, slope = 12.97°, dist_agri_m = 1,060.5 m, dist_roads_m = 425.0 m). Dominant fuels: Pastizal Inducido (25.1%) & Matorral Xerófilo (24.8%).","High-Risk Winter/Early-Spring Agricultural Burn & Cured Biomass Regime (Nov–Feb). Captures 341 post-hoc wildfires (38.93% of all fires; 5.86% hazard rate, 1.22x baseline lift) driven by cured herbaceous fuels and NOM-015 agricultural plot preparation near roads and croplands."
1,"Cluster 1 (High-Altitude Sierra Fría Forest Sanctuary — N = 3,042, 16.65%)","Centroid: PC1 = -1.34, PC2 = +3.30, PC3 = +0.00 (Pyro Index = -4.63). Highest elevation (2,765.6 m, Z = +1.64) and steep canyon slopes (30.34°, Z = +1.67), remote from agriculture (dist_agri_m = 3,967.9 m, Z = +1.79) and roads (dist_roads_m = 1,775.2 m, Z = +1.79), cool temperate microclimate (20.04°C, Z = -0.86; soil_moist_mean = 0.233). Composed 100% of temperate forest: Bosque de Pino (67.7%) & Pino-Encino (32.3%).","Baseline High-Mountain Conservation Sanctuary. Orographic thermal damping and absence of agricultural/roadside human sparks suppress post-hoc wildfire incidence to 0.66% (only 20 fires across 2 years; 2.28% of total fires, 0.14x baseline risk)."
2,"Cluster 2 (Extreme Heatwave & Drought Desiccation Apex — N = 4,695, 25.69%)","Centroid: PC1 = +2.66, PC2 = +0.11, PC3 = +0.68 (Pyro Index = +2.56). Extreme atmospheric & edaphic desiccation: peak temperature (28.62°C, Z = +1.13), extreme VPD suction (vpd_max_kpa = 2.89 kPa, Z = +1.26; vpd_72h_max_kpa = 3.05 kPa, Z = +1.26), critical minimum RH (13.54%, Z = -0.95), depleted soil moisture (0.158 m³/m³, Z = -0.98), prolonged drought (days_since_rain = 34.7 d, precip_72h = 0.68 mm), in valley/foothill interface (dist_agri_m = 1,153.3 m, elev = 2,166.5 m).","Critical Late-Spring Heatwave & Evaporative Drought Regime (Mar–Jun). Concentrates the highest wildfire hazard in the state: 388 post-hoc ignitions (44.29% of all fires; 8.26% hazard rate, 1.72x baseline lift and 12.57x higher than Cluster 1), demanding Red-Alert brigade pre-positioning."
3,"Cluster 3 (Warm Monsoon Hydrated & Green-Up Valley — N = 4,723, 25.84%)","Centroid: PC1 = -2.11, PC2 = -1.49, PC3 = +0.78 (Pyro Index = -0.62). Active summer monsoon hydration: heavy 72h rainfall (precip_72h_sum_mm = 14.46 mm, Z = +1.02; days_since_rain = 0.8 d), high relative humidity (48.69%, Z = +1.20), saturated upper soil (0.342 m³/m³, Z = +1.26), peak Sentinel-2 foliar green-up (ndmi = +0.323, Z = +1.09), low VPD (1.27 kPa), located in agricultural valleys (dist_agri_m = 1,194.0 m, elev = 2,179.9 m).","Summer Monsoon Moisture-Suppressed Valley Regime (Jul–Oct). Proves that even inside exposed agricultural valleys (PC2 = -1.49), monsoon rainfall and high foliar moisture (PC3 = +0.78, PC1 = -2.11) cut wildfire hazard down to 2.69% (127 fires; 0.56x baseline lift)."


### Step 6 — Segmentation Validation
We evaluate the mathematical and operational validity of the $K^* = 4$ segmentation across three formal criteria: **Separation**, **Consistency**, and **Practical Value** (**Deliverable 7 — Segmentation Validation Matrix**).


In [5]:
print('[DELIVERABLE 7] Segmentation Validation Matrix:')
df_validation


[DELIVERABLE 7] Segmentation Validation Matrix:


,Criterion,Assessment,Evidence
0,Separation,Strong Orthogonal Inter-Cluster Separation (Validated),"Mean Silhouette Score = 0.3039 (+21.7% over 12D raw space's 0.2496), global minimum Davies-Bouldin Index = 1.0474, and global maximum Calinski-Harabasz Index = 9,844.88 at K*=4. Pairwise Euclidean inter-centroid distances in 3D PCA space range from d(C0, C2) = 3.1225 to d(C1, C2) = 5.1618 (mean = 4.3020 SD units). One-way ANOVA confirms highly significant separation across all axes: PC1 (F = 12,477.8, p < 10^-300), PC2 (F = 10,113.8, p < 10^-300), and PC3 (F = 4,806.3, p < 10^-300)."
1,Consistency,High Intra-Cluster Cohesion & Initialization Stability (Validated),"Total 3D WCSS drops by 61.78% (from TSS = 192,692.27 to WCSS = 73,645.77), with compact mean observation-to-centroid radii across all segments (C0: 1.568 ± 0.520; C2: 1.877 ± 0.624; C3: 1.846 ± 0.781; C1: 2.453 ± 0.839). Across all N = 18,275 rows, 96.8% to 100.0% of observations in valley clusters exhibit positive sample silhouette s(i) > 0, and k-means++ (n_init=20) converges identically across random seeds."
2,Practical Value,High Operational & Ecological Relevance for CONAFOR (Validated),"Without using labels during training, K*=4 recovers ecological fuel zonation (Chi-square = 12,733.85, p < 10^-300; Cluster 1 is 100% temperate Pine/Pine-Oak forest) and stratifies post-hoc 48–72h wildfire hazard across a 12.57x dynamic range (Chi-square = 298.39, p = 2.22 x 10^-64): Cluster 2 (8.26% hazard, 44.29% of fires) and Cluster 0 (5.86% hazard, 38.93% of fires) capture 83.22% of all wildfires (729 / 876), directly guiding seasonal resource dispatch."


---
## 💡 PHASE 4: SEGMENTATION INSIGHTS AND ANALYTICAL ASSESSMENT
### Step 7 — Segmentation Insights
Integrating the unsupervised cluster topology with our prior findings from Activities 4–7 yields four major operational insights (**Deliverable 8 — Segmentation Insights Matrix**).


In [6]:
print('[DELIVERABLE 8] Segmentation Insights Matrix:')
df_insights


[DELIVERABLE 8] Segmentation Insights Matrix:


,Insight,Evidence,Practical Meaning
0,1. Bifurcation of Wildfire Hazard into Two Distinct Seasonal-Physical Regimes (Cluster 2 vs. Cluster 0),"While Activity 6 & 7 treated Quadrant IV (PC1 > 0, PC2 <= 0) as a single high-risk zone, K-Means (K*=4) splits regional wildfire risk along PC1 and PC3 into Cluster 2 (Spring Heatwave Apex: PC1=+2.66, Tmax=28.62°C, VPD=2.89 kPa, 8.26% fire rate, 44.29% of fires) and Cluster 0 (Dormant Cured-Biomass Valley: PC3=-1.19, NDMI=-0.007, Tmax=22.17°C, 5.86% fire rate, 38.93% of fires). Together they capture 83.22% of all ignitions (729 / 876).","CONAFOR cannot rely on temperature thresholds alone: winter/early-spring fires in Cluster 0 occur at moderate temperatures (22.2°C) due to cured grassland fuels (ndmi < 0) and agricultural burns, requiring NOM-015 burn regulation in Nov–Feb, whereas Cluster 2 requires tactical fire-brigade pre-deployment during Mar–Jun heatwaves."
1,2. Monsoon Hydration Neutralizes Anthropogenic Valley Proximity (Cluster 3 vs. Clusters 0 & 2),"Cluster 3 (PC1=-2.11, PC2=-1.49, PC3=+0.78) shares the exact same low-elevation agricultural interface (dist_agri_m = 1,194.0 m, elevation_m = 2,179.9 m) as Clusters 0 and 2, yet its post-hoc fire rate drops by 67.5% relative to Cluster 2 (to 2.69%) due to 72h monsoon rain (14.46 mm), high soil moisture (0.342 m³/m³), and foliar green-up (ndmi = +0.323).",Demonstrates why static spatial buffer maps (distance to agriculture/roads) produce high false-alarm rates during Jul–Oct; coupling cluster regime state with supervised ML classifiers suppresses summer false positives.
2,3. Autonomous Unsupervised Recovery of the Sierra Fría Temperate Forest Sanctuary (Cluster 1),"Operating purely on continuous PCA coordinates without categorical fuel_type or fire_ignition_48h, K-Means isolates Cluster 1 (PC2 = +3.30, elevation = 2,765.6 m, slope = 30.34°, dist_agri_m = 3,967.9 m), which post-hoc inspection reveals is 100.0% temperate forest (67.65% Bosque de Pino, 32.35% Pino-Encino) with a 0.66% fire rate (0.14x baseline).","Validates that the 3D PCA manifold preserves true eco-topographic boundaries, allowing TERRA-FIRE to assign automated low-priority baseline watch to high-mountain sanctuary voxels unless extreme drought anomalies shift a node into Cluster 2."
3,4. Orthogonal PCA Subspace Prevents Collinear Distance Distortion in Euclidean K-Means,"Clustering on the 12 standardized original variables (Max VIF = 80.10) yields a Silhouette Score of only 0.2496 at K=4 because four correlated topographic/anthropogenic variables (dist_agri_m, dist_roads_m, slope_deg, elevation_m) dominate Euclidean distances. Clustering in 3D PCA space (VIF = 1.00) boosts Silhouette by +21.7% (to 0.3039) while reducing compute dimensions by 75.0%.",Establishes a mandatory architectural rule for the TERRA-FIRE Predictive Intelligence Engine: all distance-based unsupervised segmentation and anomaly detection modules must operate in orthogonal PCA coordinates.


### Step 8 — Intelligent Segmentation Assessment & Executive Summary
1. **Number of Clusters Identified:** **$K^* = 4$ orthogonal pyro-climatic and territorial segments** discovered via `KMeans(n_clusters=4, init='k-means++', n_init=20, random_state=42)` in the 3D PCA subspace (`['PC1', 'PC2', 'PC3']`, $87.87\%$ variance retained, $\text{VIF} = 1.00$), explaining **61.78%** of 3D variance ($\text{WCSS} = 73,645.77$) with **Mean Silhouette $= 0.3039$** ($+21.7\%$ over 12D raw space), **Davies-Bouldin $= 1.0474$** (global minimum), and **Calinski-Harabasz $= 9,844.88$** (global maximum).
2. **Main Characteristics of Each Segment:**
   - **Cluster 0 (*Dormant Cured-Biomass Agricultural Valley*, $N = 5,815, 31.82\%$):** $\boldsymbol{\mu}_0 = (+0.26, -0.60, -1.19)$; lowest foliar moisture (`ndmi` $= -0.007$), cool-dry winter/early-spring weather ($22.17^\circ\text{C}$, $\text{RH} = 26.09\%$), agricultural valley proximity ($1,060.5\text{ m}$); **5.86% post-hoc fire hazard ($38.93\%$ of all fires)**.
   - **Cluster 1 (*High-Altitude Sierra Fría Forest Sanctuary*, $N = 3,042, 16.65\%$):** $\boldsymbol{\mu}_1 = (-1.34, +3.30, +0.00)$; high elevation ($2,765.6\text{ m}$), steep canyons ($30.34^\circ$), remote from agriculture ($3,967.9\text{ m}$), $100\%$ *Bosque de Pino* / *Pino-Encino*; **0.66% post-hoc fire hazard ($2.28\%$ of all fires)**.
   - **Cluster 2 (*Extreme Heatwave & Drought Desiccation Apex*, $N = 4,695, 25.69\%$):** $\boldsymbol{\mu}_2 = (+2.66, +0.11, +0.68)$; extreme temperature ($28.62^\circ\text{C}$), peak $\text{VPD}$ ($2.89\text{ kPa}$), minimum $\text{RH}$ ($13.54\%$), depleted soil moisture ($0.158\text{ m}^3/\text{m}^3$), $34.7$ rainless days; **8.26% post-hoc fire hazard ($44.29\%$ of all fires, $12.57\times$ Cluster 1)**.
   - **Cluster 3 (*Warm Monsoon Hydrated & Green-Up Valley*, $N = 4,723, 25.84\%$):** $\boldsymbol{\mu}_3 = (-2.11, -1.49, +0.78)$; active summer monsoon rain ($14.46\text{ mm/72h}$), high soil moisture ($0.342\text{ m}^3/\text{m}^3$), peak canopy greenness (`ndmi` $= +0.323$); **2.69% post-hoc fire hazard ($14.50\%$ of all fires)**.
3. **Most Relevant Insights Discovered:** Unsupervised segmentation reveals that wildfire hazard in Aguascalientes bifurcates into two distinct seasonal mechanisms—**Dormant Cured-Biomass Agricultural Burns (Cluster 0, Nov–Feb)** and **Thermodynamic Heatwave Desiccation (Cluster 2, Mar–Jun)**—which together account for **83.22% of all regional wildfires ($729 / 876$)**, while summer monsoon hydration (Cluster 3) suppresses fire rates by $67.5\%$ inside the exact same agricultural valleys.
4. **Limitations:** Spherical Euclidean $K$-Means partitions continuous meteorological transitions with hard Voronoi boundaries and relies on static topographic coordinates across the 25 monitoring nodes.
5. **Formal Readiness Assessment:** **`READY FOR ADVANCED ANALYTICS`**.
